# Clase 4: Introduccion a Random Forest
## El poder de muchos arboles tomando decisiones juntos

---

### ¿Que vamos a aprender?

1. **¿Que es un arbol de decision?** - La base de Random Forest
2. **¿Que es Random Forest?** - Muchos arboles son mejores que uno
3. **¿Como funciona?** - La intuicion paso a paso
4. **¿Cuando usarlo?** - Escenarios ideales y limitaciones
5. **Ejemplo practico** - Aplicado al problema de churn

---

## 1. Primero lo primero: ¿Que es un arbol de decision?

Antes de entender Random Forest, necesitamos entender su pieza fundamental: el **arbol de decision**.

Un arbol de decision funciona como una **serie de preguntas de si o no** que van filtrando hasta llegar a una respuesta.

### Ejemplo: ¿Este cliente hara churn?

```
                    ¿Contrato mes a mes?
                    /                 \
                  SI                   NO
                  /                     \
       ¿Tenure < 12 meses?          ¿Gasto > $80?
          /          \                /         \
        SI           NO             SI          NO
        /             \             /            \
   CHURN (85%)    ¿Soporte?    NO CHURN      NO CHURN
                   /     \       (90%)         (95%)
                 SI      NO
                 /        \
            NO CHURN    CHURN
             (60%)      (70%)
```

Es como un **diagrama de flujo** que toma decisiones basandose en los datos.

### El problema de un solo arbol

Un arbol de decision individual tiene un problema: **se memoriza los datos de entrenamiento** (sobreajuste/overfitting). Es como un estudiante que memoriza las respuestas del examen anterior pero no entiende los conceptos.

**Solucion:** ¿Y si usamos **muchos arboles** y dejamos que **voten**?

---
## 2. ¿Que es Random Forest?

Random Forest (Bosque Aleatorio) es un modelo que crea **muchos arboles de decision** y combina sus predicciones.

### Analogia: La sabiduria de la multitud

Imagina que quieres saber cuantos dulces hay en un frasco:

- Si le preguntas a **una persona**, puede equivocarse mucho
- Si le preguntas a **100 personas** y sacas el **promedio**, la respuesta sera mucho mas precisa

Random Forest aplica el mismo principio: **la opinion combinada de muchos arboles es mejor que la de uno solo**.

### ¿Como funciona paso a paso?

```
Dataset completo (7000 clientes)
        |
        v
  +-----------+-----------+-----------+
  |           |           |           |
  v           v           v           v
Muestra 1   Muestra 2   Muestra 3  ... Muestra N
(aleatorio) (aleatorio) (aleatorio)   (aleatorio)
  |           |           |           |
  v           v           v           v
Arbol 1     Arbol 2     Arbol 3    ... Arbol N
  |           |           |           |
  v           v           v           v
CHURN       NO CHURN    CHURN      ... CHURN
  |           |           |           |
  +-----+-----+-----+----+-----------+
        |
        v
   VOTACION FINAL
   (mayoria gana)
        |
        v
      CHURN
   (3 de 4 dijeron churn)
```

### Los dos trucos de Random Forest:

1. **Muestreo aleatorio de filas (Bagging):** Cada arbol se entrena con una muestra diferente del dataset (con reemplazo). Asi, cada arbol ve datos ligeramente distintos.

2. **Seleccion aleatoria de columnas:** En cada pregunta del arbol, solo se consideran algunas variables al azar. Asi, los arboles no se parecen entre si.

Estos dos trucos hacen que los arboles sean **diversos**, lo cual mejora la prediccion final.

---
## 3. ¿Cuando es bueno usar Random Forest?

### Escenarios donde Random Forest **brilla**:

| Escenario | ¿Por que? |
|-----------|----------|
| **Datos tabulares** (filas y columnas) | Es uno de los mejores modelos para tablas de datos |
| **Mezcla de variables numericas y categoricas** | Maneja ambos tipos sin problemas |
| **No sabes que variables son importantes** | El modelo selecciona las mejores automaticamente |
| **Quieres evitar sobreajuste** | El promedio de muchos arboles reduce el error |
| **Relaciones no lineales** | Captura patrones complejos que Logistic Regression no puede |
| **No quieres escalar los datos** | Funciona bien sin normalizar las variables |

### Escenarios donde Random Forest **NO es la mejor opcion**:

| Escenario | ¿Por que? |
|-----------|----------|
| **Datasets muy grandes** (millones de filas) | Entrenar muchos arboles puede ser lento |
| **Necesitas un modelo interpretable** | Es dificil explicar por que tomo una decision |
| **Prediccion en tiempo real** | Predecir con 200 arboles es mas lento que con un modelo simple |
| **Datos de texto o imagenes** | Otros modelos (redes neuronales) son mejores para esto |
| **Datos muy simples** | Si la relacion es lineal, Logistic Regression basta |

### Ejemplos del mundo real:

- **Bancos** lo usan para detectar fraude en transacciones
- **Hospitales** para predecir riesgo de enfermedades
- **E-commerce** para recomendar productos
- **Telecomunicaciones** para predecir churn
- **Seguros** para calcular primas de riesgo

---
## 4. Parametros importantes de Random Forest

Random Forest tiene parametros que podemos ajustar para mejorar el modelo:

| Parametro | ¿Que controla? | Valor tipico |
|-----------|----------------|--------------|
| `n_estimators` | Cantidad de arboles en el bosque | 50 - 500 |
| `max_depth` | Profundidad maxima de cada arbol | 5 - 30 (o None para sin limite) |
| `min_samples_split` | Minimo de datos para dividir un nodo | 2 - 10 |
| `class_weight` | Peso de cada clase (para datos desbalanceados) | 'balanced' o None |
| `random_state` | Semilla para reproducibilidad | Cualquier numero |

### Regla practica:

- **Mas arboles** (`n_estimators`) = Mejor prediccion, pero mas lento
- **Arboles mas profundos** (`max_depth`) = Captura mas detalle, pero riesgo de sobreajuste
- **`class_weight='balanced'`** = Indispensable cuando una clase es mucho menor que la otra

---
## 5. Ejemplo practico: Predecir churn con Random Forest

Vamos a entrenar un Random Forest paso a paso usando el dataset de IBM Telco Customer Churn.

In [ ]:
# Importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

sns.set_style("whitegrid")
np.random.seed(42)

print("Librerias importadas correctamente")

In [ ]:
# Cargar y limpiar el dataset
df = pd.read_csv('Telco-Customer-Churn.csv')

df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Dataset: {len(df)} clientes")
print(f"Churn: {df['Churn'].mean()*100:.1f}%")

In [ ]:
# Preparar los datos
df_encoded = pd.get_dummies(df, columns=df.select_dtypes(include='object').columns, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar datos
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Entrenamiento: {X_train_scaled.shape[0]} registros, {X_train_scaled.shape[1]} variables")
print(f"Prueba: {X_test_scaled.shape[0]} registros")

### Paso 1: Entrenar un Random Forest basico

In [ ]:
# Entrenar Random Forest con parametros basicos
inicio = time.time()
modelo_rf = RandomForestClassifier(
    n_estimators=100,          # 100 arboles
    class_weight='balanced',   # Balancear clases (importante para churn)
    random_state=42
)
modelo_rf.fit(X_train_scaled, y_train)
tiempo_entrenamiento = time.time() - inicio

print(f"Modelo entrenado en {tiempo_entrenamiento*1000:.0f} milisegundos")
print(f"Numero de arboles: {modelo_rf.n_estimators}")
print(f"Profundidad maxima: {modelo_rf.max_depth} (sin limite)")

### Paso 2: Evaluar el modelo

In [ ]:
# Hacer predicciones
y_pred = modelo_rf.predict(X_test_scaled)

# Evaluar
print("RESULTADOS DE RANDOM FOREST (100 arboles)")
print("=" * 45)
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0)}")

In [ ]:
# Matriz de confusion
fig, ax = plt.subplots(figsize=(7, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
ax.set_title('Confusion Matrix - Random Forest (100 arboles)')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Prediccion')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nInterpretacion:")
print(f"  Clientes que NO se fueron y el modelo acerto (TN): {tn}")
print(f"  Falsas alarmas - dijo churn pero no se fueron (FP): {fp}")
print(f"  No detectados - se fueron sin que el modelo lo previera (FN): {fn}")
print(f"  Bien detectados - se fueron y el modelo lo predijo (TP): {tp}")

---
## 6. La ventaja exclusiva de Random Forest: Importancia de variables

Una de las caracteristicas mas utiles de Random Forest es que nos dice **cuales variables son las mas importantes** para hacer la prediccion. Esto es valioso para el negocio porque nos dice **que factores influyen mas en el churn**.

In [ ]:
# Obtener la importancia de cada variable
importancias = pd.DataFrame({
    'Variable': X_train.columns,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=False)

# Mostrar las 10 variables mas importantes
print("TOP 10 VARIABLES MAS IMPORTANTES PARA PREDECIR CHURN")
print("=" * 55)
for i, row in importancias.head(10).iterrows():
    barra = '█' * int(row['Importancia'] * 100)
    print(f"  {row['Variable']:30s} {row['Importancia']:.4f} {barra}")

In [ ]:
# Grafico de importancia de variables (top 15)
fig, ax = plt.subplots(figsize=(10, 8))

top_15 = importancias.head(15)
ax.barh(range(len(top_15)), top_15['Importancia'].values, color='forestgreen')
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['Variable'].values)
ax.invert_yaxis()
ax.set_xlabel('Importancia')
ax.set_title('Top 15 Variables mas Importantes - Random Forest')

plt.tight_layout()
plt.show()

print("\nInterpretacion para el negocio:")
print(f"  La variable mas importante es: {importancias.iloc[0]['Variable']}")
print(f"  Esto sugiere que la empresa deberia enfocarse en esta variable")
print(f"  para disenar estrategias de retencion de clientes.")

---
## 7. Experimento: ¿Cuantos arboles necesitamos?

Vamos a entrenar Random Forest con diferente cantidad de arboles para ver como afecta al rendimiento.

In [ ]:
# Experimentar con diferente numero de arboles
n_arboles = [10, 25, 50, 100, 200, 300]
resultados = []

for n in n_arboles:
    inicio = time.time()
    modelo = RandomForestClassifier(
        n_estimators=n, class_weight='balanced', random_state=42
    )
    modelo.fit(X_train_scaled, y_train)
    tiempo = time.time() - inicio

    y_pred_temp = modelo.predict(X_test_scaled)
    resultados.append({
        'Arboles': n,
        'Accuracy': accuracy_score(y_test, y_pred_temp),
        'F1 Score': f1_score(y_test, y_pred_temp, zero_division=0),
        'Recall': recall_score(y_test, y_pred_temp, zero_division=0),
        'Tiempo (ms)': tiempo * 1000
    })

df_exp = pd.DataFrame(resultados)
print("RESULTADOS POR NUMERO DE ARBOLES")
print("=" * 60)
df_exp.round(4)

In [ ]:
# Visualizar como cambian las metricas segun el numero de arboles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metricas vs numero de arboles
axes[0].plot(df_exp['Arboles'], df_exp['Accuracy'], 'o-', label='Accuracy', color='steelblue')
axes[0].plot(df_exp['Arboles'], df_exp['F1 Score'], 's-', label='F1 Score', color='forestgreen')
axes[0].plot(df_exp['Arboles'], df_exp['Recall'], '^-', label='Recall', color='salmon')
axes[0].set_xlabel('Numero de arboles')
axes[0].set_ylabel('Valor de la metrica')
axes[0].set_title('Metricas vs Numero de Arboles')
axes[0].legend()

# Tiempo vs numero de arboles
axes[1].bar(df_exp['Arboles'].astype(str), df_exp['Tiempo (ms)'], color='forestgreen', alpha=0.7)
axes[1].set_xlabel('Numero de arboles')
axes[1].set_ylabel('Tiempo de entrenamiento (ms)')
axes[1].set_title('Tiempo de Entrenamiento vs Numero de Arboles')

plt.tight_layout()
plt.show()

print("\nObservacion: Despues de cierto punto, agregar mas arboles")
print("mejora muy poco la prediccion pero aumenta el tiempo de entrenamiento.")
print("Hay que encontrar un balance entre calidad y velocidad.")

---
## 8. Comparacion: Random Forest vs Logistic Regression vs Naive Bayes

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

# Entrenar los tres modelos y medir tiempos
modelos = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Random Forest (100)': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'Naive Bayes': GaussianNB()
}

comparacion = []
for nombre, modelo in modelos.items():
    inicio = time.time()
    modelo.fit(X_train_scaled, y_train)
    tiempo = time.time() - inicio

    y_pred_temp = modelo.predict(X_test_scaled)
    comparacion.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, y_pred_temp),
        'Precision': precision_score(y_test, y_pred_temp, zero_division=0),
        'Recall': recall_score(y_test, y_pred_temp, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred_temp, zero_division=0),
        'Tiempo (ms)': round(tiempo * 1000, 1)
    })

df_comp = pd.DataFrame(comparacion).set_index('Modelo')
print("COMPARACION DE MODELOS")
print("=" * 70)
df_comp.round(4)

In [ ]:
# Grafico comparativo
fig, ax = plt.subplots(figsize=(12, 6))

metricas = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(metricas))
ancho = 0.25

bars1 = ax.bar(x - ancho, df_comp.loc['Logistic Regression', metricas].values,
               ancho, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x, df_comp.loc['Random Forest (100)', metricas].values,
               ancho, label='Random Forest', color='forestgreen')
bars3 = ax.bar(x + ancho, df_comp.loc['Naive Bayes', metricas].values,
               ancho, label='Naive Bayes', color='salmon')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metrica')
ax.set_ylabel('Valor')
ax.set_title('Comparacion de Metricas entre Modelos')
ax.set_xticks(x)
ax.set_xticklabels(metricas)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---
## 9. Resumen: ¿Cuando elegir Random Forest?

### Usa Random Forest cuando:

- Trabajas con **datos tabulares** (tablas con filas y columnas)
- Tienes **mezcla de variables** numericas y categoricas
- Quieres saber **cuales variables son las mas importantes**
- Necesitas un modelo que **funcione bien sin mucho ajuste**
- Quieres **evitar sobreajuste** (un arbol solo tiende a memorizar)

### NO uses Random Forest cuando:

- Necesitas **velocidad maxima** (Naive Bayes o Logistic Regression son mas rapidos)
- Necesitas **explicar cada decision** al detalle (un arbol solo es mas interpretable)
- Trabajas con **texto o imagenes** (usa modelos especializados)

### Comparacion final:

| Aspecto | Logistic Regression | Random Forest | Naive Bayes |
|---------|--------------------|--------------|-----------|
| Velocidad | Rapido | Lento | Muy rapido |
| Precision general | Buena | Muy buena | Buena |
| Interpretabilidad | Alta | Media (importancia de variables) | Alta |
| Relaciones complejas | No las captura | Si las captura | No las captura |
| Requiere escalar datos | Si | No (pero no hace dano) | Si |
| Importancia de variables | No directa | Si, integrada | No directa |

---
## 10. EJERCICIO PARA LOS ESTUDIANTES

**Ejercicio 1:** Entrena un Random Forest con `max_depth=5` (arboles poco profundos) y compara los resultados con el modelo original que no tiene limite de profundidad. ¿Que cambia?

In [ ]:
# Ejercicio 1: Random Forest con max_depth=5
modelo_limitado = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,               # Limitar profundidad
    class_weight='balanced',
    random_state=42
)
modelo_limitado.fit(X_train_scaled, y_train)
y_pred_limitado = modelo_limitado.predict(X_test_scaled)

print("COMPARACION: Sin limite vs max_depth=5")
print("=" * 50)
print(f"{'Metrica':<15} {'Sin limite':>12} {'max_depth=5':>12}")
print("-" * 50)
print(f"{'Accuracy':<15} {accuracy_score(y_test, y_pred):>12.4f} {accuracy_score(y_test, y_pred_limitado):>12.4f}")
print(f"{'Precision':<15} {precision_score(y_test, y_pred, zero_division=0):>12.4f} {precision_score(y_test, y_pred_limitado, zero_division=0):>12.4f}")
print(f"{'Recall':<15} {recall_score(y_test, y_pred, zero_division=0):>12.4f} {recall_score(y_test, y_pred_limitado, zero_division=0):>12.4f}")
print(f"{'F1 Score':<15} {f1_score(y_test, y_pred, zero_division=0):>12.4f} {f1_score(y_test, y_pred_limitado, zero_division=0):>12.4f}")

print(f"\n¿Que observas? Escribe tu respuesta:")
# 

**Ejercicio 2:** Responde las siguientes preguntas:

1. Observa el grafico de importancia de variables. ¿Cuales son las 3 variables mas importantes para predecir churn? ¿Tiene sentido desde el punto de vista del negocio?

2. ¿Por que Random Forest es mas lento que Naive Bayes? ¿En que situacion valdria la pena esperar mas tiempo?

3. Si tuvieras que presentar resultados al gerente de la empresa, ¿que modelo elegirías y por que?

In [ ]:
# Espacio para las respuestas de los estudiantes

# Pregunta 1:
# 

# Pregunta 2:
# 

# Pregunta 3:
# 